# 📓 Experimento — Regressão da tarifa de táxi de NYC com Spark ML (Colab)

Este notebook replica, no Google Colab, o experimento executado no cluster Hadoop + Spark do projeto `lab-spark-ml`:

1. **Dados:** download dos arquivos mensais oficiais da NYC TLC (2024).
2. **Amostras:** limpeza e amostragem — 200.000 corridas do mês de treino e 10.000 de cada mês de predição.
3. **Etapa 1 — treino:** pipeline de preparação + LinearRegression, RandomForest e GBT, com comparação de RMSE, MAE e R².
4. **Persistência:** gravação do pipeline de preparação, dos pipelines completos de cada modelo e dos metadados.
5. **Etapa 2 — predição em batch:** carrega o pipeline salvo (sem treinar de novo) e prevê as tarifas de outros meses.
6. **Exportação:** CSVs de predições e métricas, e os modelos em `.zip`, para download.

| No cluster (`lab-spark-ml`) | Neste notebook |
|---|---|
| HDFS (`hdfs://namenode:9000/...`) | Disco do Colab (`/content/...`) |
| Spark Standalone (`spark://spark-master:7077`) | Spark local (`local[*]`) |
| `setup.sh` + `gerar_amostras.py` | Seções 1 e 2 |
| `treino_regressao.py` | Seções 3 e 4 |
| `predicao_batch.py` | Seção 5 |
| `exportar_resultados.sh` / `modelos.sh` | Seção 6 |

O processo é o mesmo, mas os números podem diferir um pouco dos obtidos no cluster: a versão do Spark e o particionamento mudam a amostragem aleatória.

## 0. Configuração do experimento

Altere aqui os meses e o tamanho das amostras. Todo o restante do notebook usa estas variáveis.

In [ ]:
ANO = 2024
MES_TREINO = 1                 # mês usado no treino
MESES_PREDICAO = [2, 6, 12]    # meses da predição em batch
N_TREINO = 200_000             # linhas da amostra de treino
N_PREDICAO = 10_000            # linhas da amostra de cada mês de predição
SEED = 42

BASE = "/content"
DIR_TLC = f"{BASE}/dados/tlc"            # arquivos mensais completos
DIR_AMOSTRAS = f"{BASE}/dados/amostras"  # amostras de treino e predição
DIR_MODELOS = f"{BASE}/modelos"          # pipelines e metadados
DIR_SAIDA = f"{BASE}/saida"              # CSVs exportados

URL_TLC = "https://d37ci6vzurychx.cloudfront.net/trip-data"


def arquivo_mes(mes):
    return f"{DIR_TLC}/yellow_tripdata_{ANO}-{mes:02d}.parquet"


def caminho_amostra(tipo, mes):
    """tipo: "treino" ou "predicao"."""
    return f"{DIR_AMOSTRAS}/{tipo}_{ANO}-{mes:02d}"


def caminho_modelo(nome):
    """nome: "preparacao", "pipeline_<modelo>", "melhor" ou "metadados"."""
    return f"{DIR_MODELOS}/{nome}"

## 1. Ambiente e download dos dados

O Colab já traz o PySpark instalado. Com `local[*]`, o Spark usa todos os núcleos da máquina do Colab como executores.

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("Experimento_Regressao_NYCTaxi")
         .master("local[*]")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.driver.memory", "6g")
         .getOrCreate())

print("Versão do Spark:", spark.version)
print("Núcleos (paralelismo):", spark.sparkContext.defaultParallelism)

Download dos arquivos mensais da TLC (cerca de 50–60 MB e ~3 milhões de corridas cada). O `-nc` evita baixar de novo o que já existe.

In [ ]:
import os

os.makedirs(DIR_TLC, exist_ok=True)

for mes in [MES_TREINO] + MESES_PREDICAO:
    nome = os.path.basename(arquivo_mes(mes))
    !wget -q -nc -P {DIR_TLC} {URL_TLC}/{nome}

!ls -lh {DIR_TLC}

## 2. Leitura, limpeza e amostragem

As mesmas funções do `preparacao.py` são usadas no treino e na predição, garantindo que os dados sejam tratados exatamente do mesmo jeito.

- **`ler_mes`:** padroniza os tipos, que variam entre os arquivos mensais, e descarta corridas com data fora do mês. Os arquivos oficiais trazem alguns registros com datas erradas, como 2002 ou 2009.
- **`limpar`:** aplica as regras do notebook da Unidade 5:
  - distância > 0;
  - tarifa entre \$2,50 (mínima oficial) e \$300;
  - desembarque depois do embarque;
  - pagamento em cartão ou dinheiro.
- **`criar_atributos`:** cria a duração da corrida, a hora do embarque, o indicador de fim de semana e o rótulo do pagamento.

In [ ]:
from itertools import chain
from pyspark.sql.functions import (col, hour, dayofweek, month, year, when,
                                   round as _round, unix_timestamp,
                                   create_map, lit, rand)

# Colunas que os pipelines salvos esperam receber
COLUNAS_MODELO = ["trip_distance", "trip_duration_min", "passenger_count",
                  "pickup_hour", "eh_fim_de_semana", "payment_label"]

MAPA_PAGAMENTO = {1: "credito", 2: "dinheiro"}


def ler_mes(mes):
    df = spark.read.parquet(arquivo_mes(mes))
    return (df
        .select(
            col("tpep_pickup_datetime").cast("timestamp"),
            col("tpep_dropoff_datetime").cast("timestamp"),
            col("passenger_count").cast("double"),
            col("trip_distance").cast("double"),
            col("fare_amount").cast("double"),
            col("payment_type").cast("int"),
        )
        .filter((year("tpep_pickup_datetime") == ANO)
                & (month("tpep_pickup_datetime") == mes))
        .withColumn("mes", lit(mes))
    )


def limpar(df):
    return (df
        .filter(col("trip_distance") > 0)
        .filter(col("fare_amount") >= 2.50)              # tarifa mínima oficial
        .filter(col("fare_amount") <= 300)               # remove casos extremos
        .filter(col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))
        .filter(col("payment_type").isin([1, 2]))        # apenas cartão e dinheiro
    )


def criar_atributos(df):
    mapping_expr = create_map(*[lit(x) for x in
        chain.from_iterable(MAPA_PAGAMENTO.items())])
    return (df
        .withColumn("trip_duration_min",
            _round((unix_timestamp("tpep_dropoff_datetime")
                  - unix_timestamp("tpep_pickup_datetime")) / 60, 2))
        .withColumn("pickup_hour", hour("tpep_pickup_datetime"))
        .withColumn("eh_fim_de_semana",
            when(dayofweek("tpep_pickup_datetime").isin([1, 7]), 1.0).otherwise(0.0))
        .withColumn("payment_label", mapping_expr[col("payment_type")])
    )

Quanto a limpeza remove do mês de treino?

In [ ]:
bruto = ler_mes(MES_TREINO)
total = bruto.count()
validas = limpar(bruto).count()
print(f"Mês {MES_TREINO:02d}: {total:,} corridas | {validas:,} válidas "
      f"({validas / total * 100:.1f}%) | removidas: {total - validas:,}")

Geração das amostras, equivalente ao `gerar_amostras.py`. A amostra é tirada **depois** da limpeza, então todas as linhas são corridas válidas.

- `sample()` reduz o volume antes do embaralhamento. A margem de 20% garante linhas suficientes.
- `orderBy(rand())` + `limit()` fixa o tamanho exato da amostra.

In [ ]:
def gerar_amostra(tipo, mes, n):
    df = limpar(ler_mes(mes))
    total = df.count()
    fracao = min(1.0, n * 1.2 / total) if total else 1.0
    amostra = (df.sample(fraction=fracao, seed=SEED)
                 .orderBy(rand(seed=SEED))
                 .limit(n))
    destino = caminho_amostra(tipo, mes)
    amostra.write.mode("overwrite").parquet(destino)
    print(f"{tipo:8} mês {mes:02d}: {min(n, total):,} de {total:,} corridas válidas -> {destino}")


gerar_amostra("treino", MES_TREINO, N_TREINO)
for mes in MESES_PREDICAO:
    gerar_amostra("predicao", mes, N_PREDICAO)

## 3. Etapa 1 — Treino

Leitura da amostra de treino, criação dos atributos e divisão 80/20 com `seed=42` para reprodutibilidade.

In [ ]:
df_feat = criar_atributos(spark.read.parquet(caminho_amostra("treino", MES_TREINO))) \
    .select(*COLUNAS_MODELO, "fare_amount")

train_df, test_df = df_feat.randomSplit([0.8, 0.2], seed=SEED)
train_df.cache()
test_df.cache()
print(f"Mês de treino: {MES_TREINO:02d}")
print(f"Treino: {train_df.count():,} | Teste: {test_df.count():,}")

df_feat.show(5)

### Pipeline de preparação

Ele é ajustado (`fit`) **apenas no treino**, para que nenhuma estatística do teste vaze para o modelo. Os estágios são:

- **Imputer:** preenche `passenger_count` nulo com a mediana do treino.
- **StringIndexer + OneHotEncoder:** transformam o tipo de pagamento em vetor binário.
- **VectorAssembler + StandardScaler:** juntam e padronizam os atributos numéricos.
- **VectorAssembler final:** produz a coluna `features`.

In [ ]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import (Imputer, StringIndexer, OneHotEncoder,
                                VectorAssembler, StandardScaler)

num_cols = ["trip_distance", "trip_duration_min", "passengers",
            "pickup_hour", "eh_fim_de_semana"]

imputer = Imputer(inputCols=["passenger_count"], outputCols=["passengers"],
                  strategy="median")
indexer = StringIndexer(inputCol="payment_label", outputCol="payment_idx",
                        handleInvalid="skip")
encoder = OneHotEncoder(inputCol="payment_idx", outputCol="payment_vec")
assembler_num = VectorAssembler(inputCols=num_cols, outputCol="features_num")
scaler = StandardScaler(inputCol="features_num",
                        outputCol="features_num_scaled",
                        withMean=True, withStd=True)
assembler_final = VectorAssembler(
    inputCols=["features_num_scaled", "payment_vec"], outputCol="features")

prep_model = Pipeline(stages=[imputer, indexer, encoder, assembler_num,
                              scaler, assembler_final]).fit(train_df)

train_prep = prep_model.transform(train_df).select("features", "fare_amount").cache()
test_prep = prep_model.transform(test_df).select("features", "fare_amount").cache()

train_prep.show(3, truncate=False)

### Treino e avaliação dos regressores

Os três modelos são avaliados no conjunto de teste do mês de treino. As métricas são:

- **RMSE:** penaliza mais os erros grandes.
- **MAE:** erro médio em dólares.
- **R²:** fração da variância da tarifa explicada pelo modelo.

In [ ]:
import time
from pyspark.ml.regression import (LinearRegression, RandomForestRegressor,
                                   GBTRegressor)
from pyspark.ml.evaluation import RegressionEvaluator

modelos = {
    "LinearRegression": LinearRegression(featuresCol="features",
                                         labelCol="fare_amount"),
    "RandomForest": RandomForestRegressor(featuresCol="features",
                                          labelCol="fare_amount",
                                          numTrees=50, seed=SEED),
    "GBT": GBTRegressor(featuresCol="features", labelCol="fare_amount",
                        maxIter=50, seed=SEED),
}

avaliadores = {m: RegressionEvaluator(labelCol="fare_amount",
                                      predictionCol="prediction",
                                      metricName=m)
               for m in ["rmse", "mae", "r2"]}

resultados = []
treinados = {}
for nome, estimador in modelos.items():
    inicio = time.time()
    modelo = estimador.fit(train_prep)
    duracao = time.time() - inicio
    pred = modelo.transform(test_prep)
    metricas = {m: ev.evaluate(pred) for m, ev in avaliadores.items()}
    treinados[nome] = modelo
    resultados.append((nome, MES_TREINO, metricas["rmse"], metricas["mae"],
                       metricas["r2"], duracao))
    print(f"{nome:16} RMSE={metricas['rmse']:.3f} | MAE={metricas['mae']:.3f} "
          f"| R2={metricas['r2']:.3f} | treino em {duracao:.1f}s")

In [ ]:
import pandas as pd

metricas_treino = pd.DataFrame(resultados, columns=["modelo", "mes_treino", "rmse",
                                                    "mae", "r2", "tempo_treino_s"])
metricas_treino.round(3)

### Quais atributos mais influenciam a tarifa?

In [ ]:
import matplotlib.pyplot as plt

nomes_features = num_cols + ["payment_vec"]
imp = pd.DataFrame({"atributo": nomes_features,
                    "importancia": treinados["RandomForest"].featureImportances.toArray()}) \
        .sort_values("importancia")

plt.figure(figsize=(7, 3.5))
plt.barh(imp["atributo"], imp["importancia"])
plt.xlabel("Importância (Random Forest)")
plt.title("Importância dos atributos na previsão da tarifa")
plt.tight_layout()
plt.show()

## 4. Persistência dos modelos e pipelines

Mesma estrutura gravada em `/modelos` no HDFS pelo `treino_regressao.py`:

| Caminho | Conteúdo |
|---|---|
| `modelos/preparacao` | Pipeline de preparação ajustado no treino |
| `modelos/pipeline_<modelo>` | Preparação + regressor, um por modelo treinado |
| `modelos/melhor` | Pipeline completo de menor RMSE |
| `modelos/metadados` | Mês de treino, melhor modelo, métricas e colunas de entrada (JSON) |

Cada pipeline completo é um `PipelineModel` aplicável direto sobre os atributos, sem precisar refazer a preparação à mão.

In [ ]:
melhor_nome = min(resultados, key=lambda r: r[2])[0]
print(f"Melhor modelo (menor RMSE): {melhor_nome}\n")

prep_model.write().overwrite().save(caminho_modelo("preparacao"))
print(f"Pipeline de preparação salvo em: {caminho_modelo('preparacao')}")

for nome, modelo in treinados.items():
    pipeline_completo = PipelineModel(stages=prep_model.stages + [modelo])
    destinos = [caminho_modelo("pipeline_" + nome.lower())]
    if nome == melhor_nome:
        destinos.append(caminho_modelo("melhor"))
    for destino in destinos:
        pipeline_completo.write().overwrite().save(destino)
        print(f"Pipeline {nome} salvo em: {destino}")

spark.createDataFrame(
    [(MES_TREINO, melhor_nome, nome, rmse, mae, r2, ",".join(COLUNAS_MODELO))
     for nome, _, rmse, mae, r2, _ in resultados],
    ["mes_treino", "melhor_modelo", "modelo", "rmse", "mae", "r2", "colunas_entrada"]) \
    .coalesce(1).write.mode("overwrite").json(caminho_modelo("metadados"))

!ls {DIR_MODELOS}

## 5. Etapa 2 — Predição em batch

A partir daqui, nada é treinado. Para simular uma segunda execução, independente da primeira, o modelo é **carregado do disco** e os metadados dizem com qual mês ele foi treinado.

Para usar outro modelo em vez do melhor, troque `MODELO_PREDICAO` por `"linearregression"`, `"randomforest"` ou `"gbt"`.

In [ ]:
MODELO_PREDICAO = "melhor"

metadados = spark.read.json(caminho_modelo("metadados")).collect()
melhor = metadados[0]["melhor_modelo"]
nome_modelo = melhor if MODELO_PREDICAO == "melhor" else \
    next(m["modelo"] for m in metadados if m["modelo"].lower() == MODELO_PREDICAO)
caminho = caminho_modelo("melhor" if MODELO_PREDICAO == "melhor"
                         else "pipeline_" + MODELO_PREDICAO)

print(f"Modelo treinado com o mês {metadados[0]['mes_treino']:02d}. Melhor no treino: {melhor}")
print(f"Usando: {nome_modelo} ({caminho})")

modelo_carregado = PipelineModel.load(caminho)
[type(s).__name__ for s in modelo_carregado.stages]

In [ ]:
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.functions import abs as _abs

amostras = [spark.read.parquet(caminho_amostra("predicao", m)) for m in MESES_PREDICAO]
df_pred = criar_atributos(reduce(DataFrame.unionByName, amostras))

predicoes = (modelo_carregado.transform(df_pred)
    .select(
        "mes",
        "tpep_pickup_datetime",
        "trip_distance",
        "trip_duration_min",
        "passenger_count",
        "payment_label",
        "fare_amount",
        _round("prediction", 2).alias("tarifa_prevista"),
    )
    .withColumn("erro_abs", _round(_abs(col("tarifa_prevista") - col("fare_amount")), 2))
    .withColumn("modelo", lit(nome_modelo))
    .orderBy("mes", "tpep_pickup_datetime")
    .cache()
)

predicoes.show(10)

### O modelo generaliza para outros meses?

Compare as métricas de cada mês com as do conjunto de teste do mês de treino. Se o erro sobe muito em algum mês, o padrão das corridas mudou: sazonalidade, trânsito ou reajuste de tarifa.

In [ ]:
avaliadores_pred = {m: RegressionEvaluator(labelCol="fare_amount",
                                           predictionCol="tarifa_prevista",
                                           metricName=m)
                    for m in ["rmse", "mae", "r2"]}
metricas_mes = []
for mes in MESES_PREDICAO:
    pred_mes = predicoes.filter(col("mes") == mes)
    valores = {m: ev.evaluate(pred_mes) for m, ev in avaliadores_pred.items()}
    metricas_mes.append((nome_modelo, mes, pred_mes.count(),
                         valores["rmse"], valores["mae"], valores["r2"]))

metricas_por_mes = pd.DataFrame(metricas_mes, columns=["modelo", "mes", "corridas",
                                                       "rmse", "mae", "r2"])
# Referência: métricas do mesmo modelo no teste do mês de treino (lidas dos metadados
# salvos, para esta seção funcionar também em uma sessão nova)
ref = next(m for m in metadados if m["modelo"] == nome_modelo)
mes_treino = ref["mes_treino"]
print(f"Referência (teste do mês {mes_treino:02d}): RMSE={ref['rmse']:.3f} | "
      f"MAE={ref['mae']:.3f} | R2={ref['r2']:.3f}")
metricas_por_mes.round(3)

In [ ]:
amostra_plot = predicoes.select("mes", "fare_amount", "tarifa_prevista").toPandas()

fig, eixos = plt.subplots(1, 2, figsize=(12, 4.5))

for mes, grupo in amostra_plot.groupby("mes"):
    eixos[0].scatter(grupo["fare_amount"], grupo["tarifa_prevista"], s=4, alpha=0.3,
                     label=f"mês {mes:02d}")
limite = amostra_plot["fare_amount"].quantile(0.99)
eixos[0].plot([0, limite], [0, limite], color="gray", linestyle="--", label="previsão perfeita")
eixos[0].set_xlim(0, limite)
eixos[0].set_ylim(0, limite)
eixos[0].set_xlabel("Tarifa real ($)")
eixos[0].set_ylabel("Tarifa prevista ($)")
eixos[0].set_title(f"Real x previsto ({nome_modelo})")
eixos[0].legend(markerscale=4)

rotulos = [f"{mes_treino:02d} (treino)"] + [f"{m:02d}" for m in metricas_por_mes["mes"]]
valores = [ref["mae"]] + list(metricas_por_mes["mae"])
barras = eixos[1].bar(rotulos, valores, color=["gray"] + ["C0"] * len(metricas_por_mes))
eixos[1].bar_label(barras, fmt="%.2f")
eixos[1].set_xlabel("Mês")
eixos[1].set_ylabel("MAE ($)")
eixos[1].set_title("Erro médio absoluto por mês")

plt.tight_layout()
plt.show()

## 6. Exportação dos resultados

No cluster, o Spark grava cada resultado como uma **pasta** com arquivos `part-*.csv`, e o `exportar_resultados.sh` junta tudo em um único arquivo. Aqui os resultados já são pequenos (algumas dezenas de milhares de linhas), então o pandas grava cada um direto em um único CSV.

In [ ]:
os.makedirs(DIR_SAIDA, exist_ok=True)

predicoes.toPandas().to_csv(f"{DIR_SAIDA}/predicoes_batch.csv", index=False)
metricas_por_mes.to_csv(f"{DIR_SAIDA}/metricas_por_mes.csv", index=False)
metricas_treino.to_csv(f"{DIR_SAIDA}/regressao_metricas.csv", index=False)

!ls -lh {DIR_SAIDA}

Download dos CSVs e dos modelos. Os modelos vão compactados em `.zip`; o arquivo equivale à pasta `modelos_salvos/` gerada pelo `./modelos.sh exportar` no cluster.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive(f"{DIR_SAIDA}/modelos", "zip", DIR_MODELOS)

for arquivo in ["predicoes_batch.csv", "metricas_por_mes.csv",
                "regressao_metricas.csv", "modelos.zip"]:
    files.download(f"{DIR_SAIDA}/{arquivo}")

### (Opcional) Guardar no Google Drive

O disco do Colab é apagado quando a sessão termina. Para rodar a etapa 2 em outra sessão sem treinar de novo:

1. Copie os modelos para o Drive com a célula abaixo.
2. Na nova sessão, aponte `DIR_MODELOS` para a pasta do Drive e execute as seções 0, 1, 2 e 5.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copytree(DIR_MODELOS, "/content/drive/MyDrive/lab-spark-ml/modelos", dirs_exist_ok=True)